# GENIE3 vs GRNBoost2: isolating the `max_features` mechanism

Follow-up to `evaluate_networksweep_final_beeline.ipynb`, which found GENIE3 and
GRNBoost2 diverging far more than expected on the 6-gene synthetic networks
(e.g. `grn_n6_e9_pos100_center`: precision_topk 0.261 vs 0.531, auprc 0.307 vs
0.567 -- with *equal* run counts on both sides, so it isn't the OOM/missing-runs
issue found separately for the dense-network families).

Candidate mechanism, from reading `arboreto/core.py`: GENIE3's RandomForest uses
`max_features='sqrt'`; GRNBoost2's gradient-boosting profile uses
`max_features=0.1`. With only 5 candidate regulators (6 genes, self excluded),
these resolve to very different numbers of regulators considered per split:

- RF: `max(1, int(sqrt(5)))` = **2** regulators/split
- GBM: `max(1, int(0.1*5))` = **1** regulator/split

(confirmed directly against fitted `sklearn` regressors: `estimators_[0].max_features_`)

GRNBoost2 compensates for its narrower per-tree feature sampling via boosting
(each new tree fits the *residual* of the previous ensemble); GENIE3's RF just
averages ~1000 independent, unpruned trees with no correction mechanism. This
notebook tests whether that difference -- not "RF vs GBM" per se -- is what's
driving the metric gap, by using `arboreto.algo.diy()` (the function both
`genie3()` and `grnboost2()` call internally) to vary `regressor_type` and
`max_features` independently.

**6-way ablation grid:**

| Config | regressor_type | max_features | Purpose |
|---|---|---|---|
| A (GENIE3 baseline)     | RF  | `'sqrt'` (-> 2/5) | current behavior |
| B                       | RF  | `1.0` (-> 5/5)    | does removing feature-starvation fix RF? |
| C                       | RF  | `1` (-> 1/5, matches GBM's count) | does RF get *worse* when starved further? |
| D (GRNBoost2 baseline)  | GBM | `0.1` (-> 1/5)    | current behavior |
| E                       | GBM | `1.0` (-> 5/5)    | does boosting stay strong with full features? |
| F                       | GBM | `'sqrt'` (-> 2/5, matches RF's count) | does GBM stay strong matched to RF's count? |

If **B closes most of the gap with D**, feature-starvation-without-correction is
the dominant cause. If **B stays far below D/E/F even with all features
available**, the gap is really about bagging (RF) vs boosting (GBM), not
`max_features`.


In [ ]:
import re
import warnings
from itertools import permutations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import auc, precision_recall_curve

from distributed import Client
from arboreto.algo import diy
from arboreto.core import RF_KWARGS, SGBM_KWARGS

warnings.filterwarnings("ignore")

INPUT_ROOT = Path("/home/gzu5140/Keerthana_b1042/TwINFER/code/Beeline/inputs/network_sweep_final")
OUTPUT_ROOT = Path("/home/gzu5140/Keerthana_b1042/TwINFER/analysis_data/network_sweep_final/beeline_inference")
RESULTS_CSV = Path("/home/gzu5140/Keerthana_b1042/TwINFER/analysis_data/network_sweep_final/ablation_max_features_test.csv")

# Dataset/family with the largest observed GENIE3-vs-GRNBoost2 gap in the main
# evaluation notebook (precision_topk 0.261 vs 0.531, auprc 0.307 vs 0.567),
# with equal run counts on both sides (no OOM/missing-run confound).
DATASET_ID = "grn_n6_e9_pos100_center_rep0"
SCHEMES = ["spread", "twin_paired"]
SIM_REPS = [0, 1, 2, 3, 4]

N_SEEDS = 5          # seeds 0..N_SEEDS-1 per (run, config) -- bump up for more statistical power
SEEDS = list(range(N_SEEDS))

print(f"Planned fits: {len(SIM_REPS) * len(SCHEMES)} runs x 6 configs x {N_SEEDS} seeds "
      f"= {len(SIM_REPS) * len(SCHEMES) * 6 * N_SEEDS} diy() calls")


In [ ]:
# --- Minimal eval helpers, copied from evaluate_networksweep_final_beeline.ipynb ---
# (unsigned/directed precision_topk/recall_topk/f1_topk/auprc only -- the two
# headline metrics the GENIE3/GRNBoost2 gap was diagnosed on)

def load_ground_truth(dataset_id: str) -> pd.DataFrame:
    gt_path = INPUT_ROOT / dataset_id / "GroundTruthNetwork.csv"
    if not gt_path.exists():
        gt_path = INPUT_ROOT / "GroundTruthNetwork.csv"
    return pd.read_csv(gt_path, header=0)


def build_edge_universe(gt_df: pd.DataFrame):
    """All directed non-self-loop gene pairs among GT genes, and the true-edge subset."""
    gt_no_self = gt_df[gt_df["Gene1"] != gt_df["Gene2"]].drop_duplicates()
    unique_nodes = sorted(set(gt_df["Gene1"]).union(set(gt_df["Gene2"])))
    possible_edges = set(permutations(unique_nodes, 2))
    true_edges = set(zip(gt_no_self["Gene1"], gt_no_self["Gene2"])) & possible_edges
    return possible_edges, true_edges


def dedupe_predictions(ranked_edges: pd.DataFrame) -> pd.DataFrame:
    """Drop self-loops; keep the highest |EdgeWeight| per (Gene1, Gene2)."""
    pred = ranked_edges[ranked_edges["Gene1"] != ranked_edges["Gene2"]].copy()
    pred["_abs"] = pred["EdgeWeight"].abs()
    return (
        pred.sort_values("_abs", ascending=False)
        .drop_duplicates(subset=["Gene1", "Gene2"])
        .reset_index(drop=True)
    )


def top_k_tie_aware_selection(predicted: pd.DataFrame, num_true_edges: int):
    """Select top-k predictions (k = num_true_edges), expanded to include ties at the
    boundary weight -- mirrors BLEval.EarlyPrecision._compute_early_precision."""
    if predicted.empty or num_true_edges == 0:
        return set(), float("nan")
    maxk = min(len(predicted), num_true_edges)
    edge_weight_topk = float(predicted.iloc[maxk - 1]["_abs"])
    nonzero = predicted.loc[predicted["_abs"] > 0, "_abs"]
    non_zero_min = float(nonzero.min()) if not nonzero.empty else 0.0
    best_val = max(non_zero_min, edge_weight_topk)
    selected = predicted[predicted["_abs"] >= best_val]
    return set(zip(selected["Gene1"], selected["Gene2"])), best_val


def precision_recall_f1(selected_edges: set, true_edges: set):
    if not selected_edges:
        return 0.0, 0.0, 0.0
    tp = len(selected_edges & true_edges)
    precision = tp / len(selected_edges)
    recall = tp / len(true_edges) if true_edges else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


def compute_auprc(ranked_edges: pd.DataFrame, gt_df: pd.DataFrame) -> float:
    """Fully threshold-free complementary metric -- mirrors BLEval.AUPRC._compute_auprc."""
    gt_genes = sorted(set(gt_df["Gene1"]).union(set(gt_df["Gene2"])))
    possible_edges = list(permutations(gt_genes, 2))
    true_edge_set = set(zip(gt_df["Gene1"], gt_df["Gene2"]))

    pred = dedupe_predictions(ranked_edges)
    pred_lookup = dict(zip(zip(pred["Gene1"], pred["Gene2"]), pred["_abs"].astype(float)))

    true_labels = [1 if e in true_edge_set else 0 for e in possible_edges]
    pred_scores = [pred_lookup.get(e, 0.0) for e in possible_edges]

    if sum(true_labels) == 0:
        return float("nan")

    precision, recall, _ = precision_recall_curve(true_labels, pred_scores)
    return float(auc(recall, precision))


def score_ranked_edges(ranked_edges: pd.DataFrame, gt_df: pd.DataFrame, num_true_edges: int, true_edges: set):
    predicted = dedupe_predictions(ranked_edges)
    selected_topk, _ = top_k_tie_aware_selection(predicted, num_true_edges)
    p, r, f1 = precision_recall_f1(selected_topk, true_edges)
    auprc = compute_auprc(ranked_edges, gt_df)
    return p, r, f1, auprc


In [ ]:
# --- Ablation grid: override only max_features, keep everything else at the
# installed arboreto version's defaults for that regressor type ---

def make_kwargs(base_kwargs: dict, max_features):
    kw = dict(base_kwargs)
    kw["max_features"] = max_features
    return kw

CONFIGS = {
    "A_RF_sqrt":  dict(regressor_type="RF",  regressor_kwargs=make_kwargs(RF_KWARGS, "sqrt")),
    "B_RF_all":   dict(regressor_type="RF",  regressor_kwargs=make_kwargs(RF_KWARGS, 1.0)),
    "C_RF_1of5":  dict(regressor_type="RF",  regressor_kwargs=make_kwargs(RF_KWARGS, 1)),
    "D_GBM_0.1":  dict(regressor_type="GBM", regressor_kwargs=make_kwargs(SGBM_KWARGS, 0.1)),
    "E_GBM_all":  dict(regressor_type="GBM", regressor_kwargs=make_kwargs(SGBM_KWARGS, 1.0)),
    "F_GBM_sqrt": dict(regressor_type="GBM", regressor_kwargs=make_kwargs(SGBM_KWARGS, "sqrt")),
}

for name, cfg in CONFIGS.items():
    print(f"{name}: regressor_type={cfg['regressor_type']!r}, max_features={cfg['regressor_kwargs']['max_features']!r}")


In [ ]:
# --- Ground truth + edge universe (fixed for this dataset_id) ---
gt_df = load_ground_truth(DATASET_ID)
possible_edges, true_edges = build_edge_universe(gt_df)
num_true_edges = len(true_edges)
print(f"{DATASET_ID}: {len(gt_df)} GT rows, {num_true_edges} true directed edges "
      f"out of {len(possible_edges)} possible")
gt_df


In [ ]:
# --- Run the ablation ---
client = Client(processes=False)
print(client)

rows = []
run_specs = [(sim_rep, scheme) for sim_rep in SIM_REPS for scheme in SCHEMES]

for sim_rep, scheme in run_specs:
    expr_path = (OUTPUT_ROOT / DATASET_ID / f"simrep{sim_rep}_{scheme}"
                 / "GRNBOOST2" / "working_dir" / "ExpressionData.csv")
    if not expr_path.exists():
        print(f"MISSING expression file, skipping: {expr_path}")
        continue

    expr_df = pd.read_csv(expr_path, sep="\t", index_col=0, header=0)
    gene_names = list(expr_df.columns)
    expr_matrix = expr_df.to_numpy()

    for config_name, cfg in CONFIGS.items():
        for seed in SEEDS:
            network = diy(
                expr_matrix,
                regressor_type=cfg["regressor_type"],
                regressor_kwargs=cfg["regressor_kwargs"],
                gene_names=gene_names,
                tf_names="all",
                client_or_address=client,
                seed=seed,
                verbose=False,
            )
            ranked_edges = network.rename(
                columns={"TF": "Gene1", "target": "Gene2", "importance": "EdgeWeight"}
            )[["Gene1", "Gene2", "EdgeWeight"]]

            p, r, f1, auprc = score_ranked_edges(ranked_edges, gt_df, num_true_edges, true_edges)

            rows.append({
                "dataset_id": DATASET_ID,
                "sim_rep": sim_rep,
                "scheme": scheme,
                "config": config_name,
                "regressor_type": cfg["regressor_type"],
                "max_features": cfg["regressor_kwargs"]["max_features"],
                "seed": seed,
                "precision_topk": p,
                "recall_topk": r,
                "f1_topk": f1,
                "auprc": auprc,
            })

    print(f"done: simrep{sim_rep}_{scheme}")

results = pd.DataFrame(rows)
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(RESULTS_CSV, index=False)
print(f"\nSaved {len(results)} rows to {RESULTS_CSV}")
results.head()


In [ ]:
# --- Summary: mean/std per config ---
CONFIG_ORDER = ["A_RF_sqrt", "B_RF_all", "C_RF_1of5", "D_GBM_0.1", "E_GBM_all", "F_GBM_sqrt"]

summary = (
    results.groupby("config")[["precision_topk", "recall_topk", "f1_topk", "auprc"]]
    .agg(["mean", "std"])
    .reindex(CONFIG_ORDER)
)
summary


In [ ]:
# --- Interpretation helper: how much of the A-D gap does each RF variant close? ---
mean_p = results.groupby("config")["precision_topk"].mean()
mean_auprc = results.groupby("config")["auprc"].mean()

gap_precision = mean_p["D_GBM_0.1"] - mean_p["A_RF_sqrt"]
gap_auprc = mean_auprc["D_GBM_0.1"] - mean_auprc["A_RF_sqrt"]

closed_by_B_precision = (mean_p["B_RF_all"] - mean_p["A_RF_sqrt"]) / gap_precision if gap_precision else float("nan")
closed_by_B_auprc = (mean_auprc["B_RF_all"] - mean_auprc["A_RF_sqrt"]) / gap_auprc if gap_auprc else float("nan")

print(f"Baseline gap (D - A): precision_topk={gap_precision:+.3f}, auprc={gap_auprc:+.3f}")
print(f"Fraction of gap closed by B (RF, max_features=1.0 i.e. all regulators):")
print(f"  precision_topk: {closed_by_B_precision:.0%}")
print(f"  auprc:          {closed_by_B_auprc:.0%}")
print()
print("If these are close to 100%: max_features/feature-starvation is the dominant cause.")
print("If these are close to 0%: it's bagging (RF) vs boosting (GBM), not max_features.")


In [ ]:
# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, metric, title in zip(axes, ["precision_topk", "auprc"], ["Precision (top-k, tie-aware)", "AUPRC"]):
    data = [results.loc[results["config"] == c, metric].values for c in CONFIG_ORDER]
    bp = ax.boxplot(data, labels=CONFIG_ORDER, patch_artist=True)
    colors = ["#4C78A8"] * 3 + ["#F58518"] * 3  # RF configs blue, GBM configs orange
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(title)
    ax.set_xticklabels(CONFIG_ORDER, rotation=30, ha="right")
    ax.set_ylim(0, 1)
    ax.axhline(mean_p["A_RF_sqrt"] if metric == "precision_topk" else mean_auprc["A_RF_sqrt"],
               color="#4C78A8", linestyle="--", linewidth=1, alpha=0.7)
    ax.axhline(mean_p["D_GBM_0.1"] if metric == "precision_topk" else mean_auprc["D_GBM_0.1"],
               color="#F58518", linestyle="--", linewidth=1, alpha=0.7)

fig.suptitle(f"{DATASET_ID}: max_features ablation (RF=blue, GBM=orange; dashed lines = A/D baselines)")
fig.tight_layout()
plt.show()
